# pinn_dispersion.ipynb — Nonlinear Dispersion Curve via PINN Simulation

Uses the segment-by-segment PINN solver (`pinn_ndof_chain.py`) to simulate a **periodic ring**
of N meta-impactor unit cells, then extracts the full nonlinear dispersion relation ω(k)
— including **multi-band effects** — via a 2-D FFT.

## Why this is the "real" dispersion curve

Unlike `dispersion_curve.py` (RK4 integration), here:
- Between-impact dynamics are solved by a PINN that minimises the ODE residual at collocation points  
- Impact times are found by exact root-finding on the frozen network  
- The stitched spatiotemporal field `x_n(t)` carries both the primary response and the
  nonlinear energy injected by each impact event

## Multi-band physics

A free-flying impactor of mass `my` bouncing inside a gap `D` acts as a **nonlinear
local resonator**.  In k–ω space this produces:

| Feature | Origin |
|---------|---------|
| **Acoustic branch** | Primary-mass lattice wave (same as linear chain) |
| **Flat optical-like branch** | Effective resonance of impactor at `ω_eff = 2π / T_impact` |
| **Band gap** | Destructive interference between acoustic and impact-resonance bands |
| **Higher harmonics** | Nonlinear energy transfer at `2ω_eff`, `3ω_eff`, … |

## Pipeline
1. Periodic ring K matrix (cyclic tridiagonal)  
2. Broadband random ICs (excites all wavenumbers simultaneously)  
3. Two-phase PINN per segment → stitched spatiotemporal array `x_n(t)`  
4. Resample to uniform time grid (dt fine enough to resolve higher bands)  
5. 2-D FFT → S(k, ω) heatmap + DOS profile + ridge extraction

In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.ticker import MaxNLocator
from scipy.interpolate import interp1d
import os

from pinn_ndof_chain import PIPNNs, find_impact_times, propagate_ics
from dispersion_curve import dispersion_from_2dfft, plot_dispersion_heatmap, \
                             linear_dispersion, extract_ridge

os.environ['CUDA_VISIBLE_DEVICES'] = '0'
np.random.seed(1234)
tf.compat.v1.set_random_seed(1234)
print('TF version:', tf.__version__)

## Physical parameters

Same values as `pinn_ndof_chain_sim.ipynb`.  
The stiffness matrix is **cyclic tridiagonal** (periodic ring), which is required for a well-defined dispersion relation.

In [ ]:
N    = 10       # number of unit cells (= n_dof)
mx   = 1.0      # primary mass
my   = 0.1      # impactor mass
k    = 100.0    # inter-cell spring stiffness
c    = 0.5      # damping coefficient
D    = 1.0      # impact gap
r    = 0.7      # coefficient of restitution

# ── Periodic (ring) stiffness matrix ──────────────────────────────────────
# K_ring[i,i] = 2k  for all i  (not k for the end cells)
# K_ring[i,(i±1)%N] = -k       (wraps around)
K_ring = np.zeros((N, N))
for i in range(N):
    K_ring[i, i]           = 2.0 * k
    K_ring[i, (i - 1) % N] = -k
    K_ring[i, (i + 1) % N] = -k

M_mat = np.diag(mx * np.ones(N))
C_mat = (c / k) * K_ring          # proportional damping

# No external forcing — broadband ICs excite all modes
phi1 = np.array([[0.0]])
phi2 = np.array([[1.0]])           # unused when phi1=0

print('K_ring (diag)  :', np.diag(K_ring))
print('K_ring (off-d1):', np.diag(K_ring, 1))
print('K_ring [0,-1]  :', K_ring[0, -1])   # should be -k (ring closure)

## PINN hyper-parameters

In [ ]:
layers              = [1, 64, 64, N]
hyp_ini_weight_loss = np.array([100.0, 1.0])   # [beta_icx, beta_fx]
optimizer_LB_value  = True
lb = np.array([0.0])
ub = np.array([4.0])

## Initial conditions — broadband random perturbation

Small random displacements excite all spatial wavenumbers simultaneously,
so the 2-D FFT will reveal the full dispersion band.

In [ ]:
rng = np.random.default_rng(42)

x0_init  = rng.uniform(-0.05, 0.05, (1, N)).astype(np.float32)
xt0_init = np.zeros((1, N), dtype=np.float32)

y0_init  = np.zeros(N)           # impactors start flush with primary masses
yt0_init = np.full(N, -0.5)      # all impactors move downward initially

# Velocity-update matrix  A^{-1} B  (momentum + restitution)
A_mat   = np.array([[mx, my], [1.0, -1.0]])
B_mat   = np.array([[mx, my], [-r,   r  ]])
A_inv_B = np.linalg.inv(A_mat) @ B_mat

print('x0_init  :', np.round(x0_init, 4))
print('yt0_init :', yt0_init)

## Multi-segment PINN simulation

Each segment:
1. Train PINN on `[0, T_seg]` → frozen network  
2. Root-find impact times (Phase 2)  
3. Record `x_n(t)` up to the first impact  
4. Apply velocity update, propagate ICs to next segment  

Stop when accumulated time reaches `T_total`.

> **Note:** Each segment requires one full PINN training.  
> With `nIter=500` and L-BFGS polishing, expect ~2–5 min per segment.

In [ ]:
T_total   = 20.0    # total simulation time (s)  — increase for finer ω resolution
T_seg_max = 4.0     # maximum window per segment
num_pts   = 500     # collocation / output points per segment
nIter     = 500     # Adam iterations per segment

t0_arr = np.array([[0.0]])

# Running state
cur_x0   = x0_init.copy()
cur_xt0  = xt0_init.copy()
cur_y0   = y0_init.copy()
cur_yt0  = yt0_init.copy()
cur_phi  = np.array([[0.0]])
t_cum    = 0.0

# Storage
all_t_segs = []    # list of 1-D arrays (global time)
all_x_segs = []    # list of (num_pts+1, N) arrays

seg_idx = 0
while t_cum < T_total:
    seg_idx += 1
    T_win = min(T_seg_max, T_total - t_cum)
    if T_win < 1e-3:
        break

    print(f"\n{'='*60}")
    print(f"  Segment {seg_idx}  |  t_cum = {t_cum:.4f} s  |  window = {T_win:.4f} s")
    print(f"{'='*60}")

    t_arr = np.linspace(0.0, T_win, num_pts).reshape(-1, 1)

    model = PIPNNs(
        lb, ub,
        t0_arr, t_arr,
        cur_x0, cur_xt0,
        cur_y0, cur_yt0,
        M_mat, K_ring, D, N,
        cur_phi, phi1, phi2,
        layers, hyp_ini_weight_loss,
        C=C_mat,
        optimizer_LB=optimizer_LB_value,
    )
    model.train(nIter=nIter, optimizer_LB=optimizer_LB_value)

    t_impacts, hit = find_impact_times(model, cur_y0, cur_yt0, D, T_win)

    if hit.any():
        first_dof = int(np.argmin(t_impacts))
        t_end     = float(t_impacts[first_dof])
        print(f"  Impact: DOF {first_dof+1} at t = {t_end:.5f} s")
    else:
        t_end = T_win
        print(f"  No impact detected; using full window t_end = {t_end:.4f} s")

    # Predict on a fine uniform grid within this segment
    t_sim  = np.linspace(0.0, t_end, num_pts + 1).reshape(-1, 1)
    x_sim, xt_sim, _ = model.predict(t_sim)

    all_t_segs.append(t_sim.flatten() + t_cum)
    all_x_segs.append(x_sim)            # (num_pts+1, N)

    # Propagate ICs
    if hit.any():
        x_end_pred, xt_end_pred, _ = model.predict(np.array([[t_end]]))
        cur_x0, cur_xt0, cur_y0, cur_yt0 = propagate_ics(
            model, t_end,
            x_end_pred, xt_end_pred,
            cur_y0, cur_yt0,
            first_dof,
            mx * np.ones(N), my * np.ones(N), r,
            A_inv_B,
        )
    else:
        x_end_pred, xt_end_pred, _ = model.predict(np.array([[t_end]]))
        cur_x0  = x_end_pred
        cur_xt0 = xt_end_pred
        cur_y0  = cur_y0 + cur_yt0 * t_end
        # cur_yt0 unchanged

    t_cum   += t_end
    cur_phi  = cur_phi + t_end

print(f"\nSimulation complete: {seg_idx} segments, T_total = {t_cum:.4f} s")

## Stitch segments → uniform spatiotemporal array

Concatenate all segments and resample onto a **uniform** time grid required by the 2-D FFT.
Interpolation uses linear splines (sufficient for smooth PINN outputs).

In [ ]:
# Concatenate all segments (boundary duplicates are fine for interpolation)
t_raw = np.concatenate(all_t_segs)           # (M_total,)
X_raw = np.vstack(all_x_segs).T              # (N, M_total)

# ── Estimate effective impact frequency from segment lengths ───────────────
seg_lengths   = np.array([segs[-1] - segs[0] for segs in all_t_segs])
T_impact_mean = float(np.mean(seg_lengths))
omega_eff     = 2.0 * np.pi / T_impact_mean   # effective impactor resonance frequency

omega_max_lin = 2.0 * np.sqrt(k / mx)         # top of linear acoustic branch (rad/s)

# ── Uniform time grid ──────────────────────────────────────────────────────
# dt must resolve at least the 4th harmonic of omega_eff OR omega_max_lin,
# whichever is higher — with at least 10 points per shortest cycle.
n_bands_shown = 4
omega_resolve  = max(n_bands_shown * omega_eff, omega_max_lin)
dt_target      = (2.0 * np.pi / omega_resolve) / 10.0
t_uniform      = np.arange(t_raw[0], t_raw[-1], dt_target)

X_uniform = np.zeros((N, len(t_uniform)))
for i in range(N):
    f_interp         = interp1d(t_raw, X_raw[i, :], kind='linear', fill_value='extrapolate')
    X_uniform[i, :]  = f_interp(t_uniform)

print(f'Mean impact period   : {T_impact_mean:.4f} s')
print(f'Effective ω_impact   : {omega_eff:.4f} rad/s')
print(f'Linear ω_max         : {omega_max_lin:.4f} rad/s')
print(f'Spatiotemporal array : {X_uniform.shape}   (N_dof × N_time)')
print(f'Time span            : {t_uniform[0]:.4f} → {t_uniform[-1]:.4f} s')
print(f'dt                   : {dt_target:.5f} s')
print(f'ω resolution Δω      : {2*np.pi/(t_uniform[-1]-t_uniform[0]):.4f} rad/s')

In [ ]:
k_pos, omega_pos, spectrum = dispersion_from_2dfft(
    t_uniform, X_uniform, skip_transient=0.15)

print(f'k range  : {k_pos[0]:.4f} → {k_pos[-1]:.4f} rad/unit-cell')
print(f'ω range  : {omega_pos[0]:.4f} → {omega_pos[-1]:.4f} rad/s')
print(f'Spectrum : {spectrum.shape}')

## Full-band dispersion heatmap S(k, ω)

The ω axis is extended to `n_bands_shown × ω_eff` to reveal all bands.

- **White dashed** — linear acoustic branch `ω_lin(k) = 2√(K/M)|sin(k/2)|`  
- **Cyan dotted** — effective impact resonance `ω_eff = 2π / T_impact` and its harmonics  
- Bright ridges above `ω_lin_max` are **higher-order bands** generated by nonlinear impacts

In [ ]:
mpl.rcParams.update({
    'font.family':      'Times New Roman',
    'mathtext.fontset': 'custom',
    'mathtext.rm':      'Times New Roman',
    'mathtext.it':      'Times New Roman:italic',
    'mathtext.bf':      'Times New Roman:bold',
    'pdf.fonttype':     42,
    'ps.fonttype':      42,
})
FS = 20
LW = 2.0

# ── Clip spectrum to multi-band ω range ────────────────────────────────────
omega_max_plot = n_bands_shown * max(omega_eff, omega_max_lin)
i_om_max       = np.searchsorted(omega_pos, omega_max_plot)
omega_plot     = omega_pos[:i_om_max]
spec_plot      = spectrum[:, :i_om_max]

S_dB = 10.0 * np.log10(spec_plot + 1e-30)
S_dB -= S_dB.max()
S_dB  = np.clip(S_dB, -40.0, 0.0)

k_norm = k_pos / np.pi    # [0, 1]

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.pcolormesh(k_norm, omega_plot, S_dB.T,
                   cmap='inferno', shading='auto',
                   vmin=-40.0, vmax=0.0)
cbar = fig.colorbar(im, ax=ax, pad=0.02)
cbar.set_label('Spectral power (dB)', fontsize=FS - 4)
cbar.ax.tick_params(labelsize=FS - 6)

# Linear acoustic branch
k_line  = np.linspace(0, np.pi, 300)
om_line = linear_dispersion(k_line, k, mx)
ax.plot(k_line / np.pi, om_line, 'w--', lw=LW,
        label=r'Linear  ($D\!\to\!\infty$)')

# Effective impact resonance harmonics
for n_harm in range(1, n_bands_shown + 1):
    om_h = n_harm * omega_eff
    if om_h <= omega_max_plot:
        ax.axhline(om_h, color='cyan', lw=1.2, linestyle=':',
                   label=(rf'$\omega_{{eff}}$  ($T_{{imp}}={T_impact_mean:.2f}$ s)'
                          if n_harm == 1 else
                          rf'${n_harm}\,\omega_{{eff}}$'))

ax.set_xlabel(r'Wavenumber  $k/\pi$',         fontsize=FS, labelpad=8)
ax.set_ylabel(r'Frequency  $\omega$  (rad/s)', fontsize=FS, labelpad=10)
ax.set_xlim(0, 1)
ax.set_ylim(0, omega_max_plot)
ax.xaxis.set_major_locator(MaxNLocator(nbins=5, prune='both'))
ax.yaxis.set_major_locator(MaxNLocator(nbins=8, prune='both'))
ax.tick_params(axis='both', labelsize=FS - 2)
ax.legend(fontsize=FS - 6, loc='upper left', framealpha=0.6)
ax.set_title(f'PINN dispersion  —  {N}-DOF ring  (D={D}, r={r}, my/mx={my/mx})',
             fontsize=FS - 4)

plt.tight_layout(pad=1.5)
plt.savefig('pinn_dispersion_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

print(f'ω_eff     : {omega_eff:.3f} rad/s  (impact resonance)')
print(f'ω_lin_max : {omega_max_lin:.3f} rad/s  (linear acoustic top)')
print(f'ω_max_plot: {omega_max_plot:.3f} rad/s')

## Density of States (DOS) — band gap identification

Integrating S(k, ω) over all k gives the **DOS(ω)**: a 1-D profile where  
- peaks → pass bands  
- valleys / zeros → **band gaps**

This is the clearest way to read off the gap frequencies without ambiguity.

In [ ]:
from scipy.signal import find_peaks

# ── DOS: sum spectrum over all k ───────────────────────────────────────────
dos = spec_plot.sum(axis=0)          # shape (n_omega,)
dos_dB = 10.0 * np.log10(dos + 1e-30)
dos_dB -= dos_dB.max()

# ── Find band-gap valleys (significant dips in DOS) ───────────────────────
neg_dos = -dos_dB                    # invert so valleys become peaks
gap_idx, gap_props = find_peaks(neg_dos, height=5.0, distance=5)   # >5 dB dip
gap_freqs = omega_plot[gap_idx]

# ── Find pass-band peaks ───────────────────────────────────────────────────
band_idx, _ = find_peaks(dos_dB, height=-30.0, distance=5)
band_freqs  = omega_plot[band_idx]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: DOS curve
ax = axes[0]
ax.plot(omega_plot, dos_dB, 'k-', lw=1.5, label='DOS')
for gf in gap_freqs:
    ax.axvline(gf, color='red',  lw=1.2, linestyle='--', alpha=0.8)
for bf in band_freqs:
    ax.axvline(bf, color='blue', lw=0.8, linestyle=':',  alpha=0.6)
ax.axvline(omega_eff,     color='cyan', lw=1.5, linestyle='-.', label=r'$\omega_{eff}$')
ax.axvline(omega_max_lin, color='gray', lw=1.5, linestyle='-.', label=r'$\omega_{lin,max}$')
ax.set_xlabel(r'$\omega$  (rad/s)', fontsize=FS)
ax.set_ylabel('DOS  (dB, normalised)', fontsize=FS)
ax.set_xlim(0, omega_max_plot)
ax.set_ylim(-42, 2)
ax.tick_params(labelsize=FS - 3)
ax.legend(fontsize=FS - 6)
ax.set_title('Density of States', fontsize=FS - 3)

# Right: k-averaged spectrum (same as DOS but as image slice)
ax2 = axes[1]
ax2.plot(dos_dB, omega_plot, 'k-', lw=1.5)
ax2.fill_betweenx(omega_plot, dos_dB, -42, alpha=0.25, color='steelblue')
for gf in gap_freqs:
    ax2.axhline(gf, color='red',  lw=1.2, linestyle='--',
                label=f'gap ~ {gf:.2f} rad/s')
ax2.set_ylabel(r'$\omega$  (rad/s)', fontsize=FS)
ax2.set_xlabel('DOS  (dB)', fontsize=FS)
ax2.set_ylim(0, omega_max_plot)
ax2.set_xlim(-42, 2)
ax2.tick_params(labelsize=FS - 3)
if len(gap_freqs):
    ax2.legend(fontsize=FS - 6)
ax2.set_title('DOS  (horizontal)', fontsize=FS - 3)

plt.tight_layout()
plt.savefig('pinn_dispersion_dos.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nPass-band centre frequencies (rad/s): {np.round(band_freqs, 3)}")
print(f"Band-gap centre frequencies  (rad/s): {np.round(gap_freqs,  3)}")

## Multi-band ridge extraction

Extract the dominant ω at each k **independently within each pass-band window**
(defined by the gap frequencies found above).  This separates the bands cleanly
and avoids the ridge-picker jumping between branches.

In [ ]:
def extract_ridge_in_band(k_pos, omega_plot, spec_plot, omega_lo, omega_hi):
    """
    Extract dominant ω for each k within the frequency window [omega_lo, omega_hi].
    Returns (k_pos, omega_ridge) where omega_ridge[i] = NaN if no peak found.
    """
    i_lo = np.searchsorted(omega_plot, omega_lo)
    i_hi = np.searchsorted(omega_plot, omega_hi)
    if i_hi <= i_lo:
        return k_pos, np.full(len(k_pos), np.nan)
    omega_ridge = np.full(len(k_pos), np.nan)
    for i in range(len(k_pos)):
        row = spec_plot[i, i_lo:i_hi]
        if row.max() > 0:
            omega_ridge[i] = omega_plot[i_lo + int(np.argmax(row))]
    return k_pos, omega_ridge

# ── Define band windows from DOS gap frequencies ───────────────────────────
# band edges: [0, gap1, gap2, ..., omega_max_plot]
band_edges = np.concatenate([[0.5], np.sort(gap_freqs), [omega_max_plot]])

palette = plt.cm.plasma(np.linspace(0.1, 0.9, len(band_edges) - 1))

fig, ax = plt.subplots(figsize=(8, 6))

# Linear baseline
k_line  = np.linspace(0, np.pi, 300)
om_line = linear_dispersion(k_line, k, mx)
ax.plot(k_line / np.pi, om_line, 'k--', lw=LW,
        label=r'Linear  ($D\!\to\!\infty$)')

# Per-band ridge
for b_idx in range(len(band_edges) - 1):
    lo, hi = band_edges[b_idx], band_edges[b_idx + 1]
    _, omega_ridge_b = extract_ridge_in_band(k_pos, omega_plot, spec_plot, lo, hi)
    valid = ~np.isnan(omega_ridge_b)
    if valid.any():
        ax.plot(k_pos[valid] / np.pi, omega_ridge_b[valid],
                'o-', color=palette[b_idx], lw=LW, ms=6,
                label=f'Band {b_idx + 1}  ({lo:.1f}–{hi:.1f} rad/s)')

# Shade band-gap regions
for gf in gap_freqs:
    ax.axhspan(gf - 0.5, gf + 0.5, color='red', alpha=0.12)

ax.axhline(omega_eff,     color='cyan', lw=1.2, linestyle='-.', label=r'$\omega_{eff}$')
ax.axhline(omega_max_lin, color='gray', lw=1.2, linestyle='-.', label=r'$\omega_{lin,max}$')

ax.set_xlabel(r'Wavenumber  $k/\pi$',         fontsize=FS, labelpad=8)
ax.set_ylabel(r'Frequency  $\omega$  (rad/s)', fontsize=FS, labelpad=10)
ax.set_xlim(0, 1)
ax.set_ylim(0, omega_max_plot)
ax.xaxis.set_major_locator(MaxNLocator(nbins=5, prune='both'))
ax.yaxis.set_major_locator(MaxNLocator(nbins=8, prune='both'))
ax.tick_params(axis='both', labelsize=FS - 2)
ax.legend(fontsize=FS - 6, loc='upper left', framealpha=0.85)
ax.set_title(f'Multi-band dispersion  —  {N} DOFs  (D={D}, r={r})', fontsize=FS - 4)

plt.tight_layout(pad=1.5)
plt.savefig('pinn_dispersion_multiband.png', dpi=300, bbox_inches='tight')
plt.show()

# ── Summary table ──────────────────────────────────────────────────────────
print(f"\n{'Band':>6}  {'ω window (rad/s)':>22}  {'k/π pts with ridge':>20}")
for b_idx in range(len(band_edges) - 1):
    lo, hi = band_edges[b_idx], band_edges[b_idx + 1]
    _, ridge_b = extract_ridge_in_band(k_pos, omega_plot, spec_plot, lo, hi)
    n_valid = int((~np.isnan(ridge_b)).sum())
    print(f"{b_idx+1:>6}  {lo:>10.2f} – {hi:<10.2f}  {n_valid:>20}")